# SVM Hyperparameter Tuning — Audio Features

This notebook tunes the SVM baseline (exact and Nyström approximated) over the audio-feature split, analyzing C/γ, approximation size, and SGD hyperparameters.

In [1]:
from pathlib import Path
import logging
from itertools import product

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import accuracy_score, roc_auc_score
from tqdm.auto import tqdm

from ml.data import (
    AUDIO_FEATURES,
    RANK_COLUMN,
    create_classification_splits,
    load_classification_dataframe,
)
from ml.models import build_svm
from ml.train import train_model, plot_training_history


/Users/hallaei/miniconda3/envs/aml_cwl/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

In [3]:
DATA_PATH = Path('data/datasets/Spotify_Dataset_V3.csv')
TOP_K = 10
FEATURE_SET_NAME = 'Audio Features'
HOLDOUT_FRACTION = 0.4
DEV_SHARE = 0.5
RANDOM_STATE = 51
MAX_EPOCHS = 15
EARLY_STOPPING_PATIENCE = 3
EARLY_STOPPING_MIN_DELTA = 1e-3

feature_cols = AUDIO_FEATURES


In [4]:
classification_df, target_col = load_classification_dataframe(
    DATA_PATH, feature_cols, RANK_COLUMN, TOP_K, FEATURE_SET_NAME
)
train_df, dev_df, test_df = create_classification_splits(
    classification_df,
    target_col,
    HOLDOUT_FRACTION,
    DEV_SHARE,
    RANDOM_STATE,
)
print(
    f'Train={len(train_df)} Dev={len(dev_df)} Test={len(test_df)} | '
    f'Pos rates -> Train {train_df[target_col].mean():.2%}, '
    f'Dev {dev_df[target_col].mean():.2%}, Test {test_df[target_col].mean():.2%}'
)


2025-11-19 23:39:35,617 [INFO] Loading dataset from data/datasets/Spotify_Dataset_V3.csv


Engineering Audio Features...


2025-11-19 23:39:37,834 [INFO] Prepared dataframe with 651936 songs (positive rate 4.82%).
2025-11-19 23:39:37,873 [INFO] Creating splits with holdout_fraction=0.40 and dev_share=0.50
2025-11-19 23:39:38,141 [INFO] Split sizes -> train=395328, dev=126410, test=130198
2025-11-19 23:39:38,159 [INFO] Train split positive rate: 4.73%
2025-11-19 23:39:38,165 [INFO] Dev split positive rate: 4.81%
2025-11-19 23:39:38,188 [INFO] Test split positive rate: 5.08%


Train=395328 Dev=126410 Test=130198 | Pos rates -> Train 4.73%, Dev 4.81%, Test 5.08%


In [5]:
dataset_splits = {
    'Train': (train_df[feature_cols], train_df[target_col]),
    'Dev': (dev_df[feature_cols], dev_df[target_col]),
    'Test': (test_df[feature_cols], test_df[target_col]),
}
classes = sorted(train_df[target_col].unique())


## Hyperparameter grid

In [6]:
approximation_options = [True, False]
c_options = [0.1, 1.0, 10.0]
gamma_options = ['scale', 0.01, 0.1]
sgd_alpha_options = [1e-5, 1e-4, 1e-3]
sgd_max_iter_options = [1000, 2000]
n_components_options = [300, 600, 900]

search_space = []
for approximate, C, gamma, sgd_alpha, sgd_max_iter, n_components in product(
    approximation_options,
    c_options,
    gamma_options,
    sgd_alpha_options,
    sgd_max_iter_options,
    n_components_options,
):
    if not approximate and gamma == 'scale':
        valid_gammas = ['scale']
    else:
        valid_gammas = [gamma]
    for gamma_val in valid_gammas:
        search_space.append({
            'approximate': approximate,
            'C': C,
            'gamma': gamma_val,
            'sgd_alpha': sgd_alpha,
            'sgd_max_iter': sgd_max_iter,
            'n_components': n_components,
        })

len(search_space)


324

## Helper functions

In [7]:
def build_svm_model(random_state, config):
    model = build_svm(
        random_state=random_state,
        c=config['C'],
        gamma=config['gamma'],
        approximate=config['approximate'],
        n_components=config['n_components'],
        sgd_alpha=config['sgd_alpha'],
        sgd_max_iter=config['sgd_max_iter'],
    )
    return model


def evaluate_dev(model, dev_data):
    X_dev, y_dev = dev_data
    y_pred = model.predict(X_dev)
    dev_acc = accuracy_score(y_dev, y_pred)
    scores = model.decision_function(X_dev)
    dev_auc = roc_auc_score(y_dev, scores)
    return dev_acc, dev_auc


## Run sweep

In [ ]:
results = []
histories = {}
progress = tqdm(search_space, desc='SVM configs')
for config in progress:
    config_name = (
        f"approx={config['approximate']} C={config['C']} gamma={config['gamma']} "
        f"sgd_alpha={config['sgd_alpha']} sgd_iter={config['sgd_max_iter']} "
        f"n_comp={config['n_components']}"
    )
    progress.set_postfix_str(config_name)
    model = build_svm_model(RANDOM_STATE, config)
    trained_model, history_df = train_model(
        model,
        dataset_splits['Train'],
        dataset_splits['Dev'],
        epochs=MAX_EPOCHS,
        classes=classes,
        early_stopping_patience=EARLY_STOPPING_PATIENCE,
        early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,
    )
    dev_acc, dev_auc = evaluate_dev(trained_model, dataset_splits['Dev'])
    best_val_loss = history_df['val_loss'].min()
    results.append({
        **config,
        'best_val_loss': best_val_loss,
        'dev_accuracy': dev_acc,
        'dev_auc': dev_auc,
        'epochs_run': len(history_df),
    })
    histories[config_name] = history_df

results_df = pd.DataFrame(results).sort_values('best_val_loss').reset_index(drop=True)
results_df.head()


SVM configs:   0%|          | 0/324 [00:00<?, ?it/s, approx=True C=0.1 gamma=scale sgd_alpha=1e-05 sgd_iter=1000 n_comp=300]2025-11-19 23:40:18,185 [INFO] Epoch 1/15 - loss: 0.6270 - val_loss: 0.7093 - accuracy: 0.6542 - val_accuracy: 0.5884 - auc: 0.7729 - val_auc: 0.5600
2025-11-19 23:42:56,243 [INFO] Epoch 2/15 - loss: 0.6270 - val_loss: 0.7093 - accuracy: 0.6542 - val_accuracy: 0.5884 - auc: 0.7729 - val_auc: 0.5600
2025-11-19 23:45:00,028 [INFO] Epoch 3/15 - loss: 0.6270 - val_loss: 0.7093 - accuracy: 0.6542 - val_accuracy: 0.5884 - auc: 0.7729 - val_auc: 0.5600
2025-11-19 23:46:21,111 [INFO] Epoch 4/15 - loss: 0.6270 - val_loss: 0.7093 - accuracy: 0.6542 - val_accuracy: 0.5884 - auc: 0.7729 - val_auc: 0.5600
2025-11-19 23:46:21,139 [INFO] Early stopping triggered after 4 epochs (no val_loss improvement for 3 epochs).
SVM configs:   0%|          | 1/324 [06:45<36:20:47, 405.10s/it, approx=True C=0.1 gamma=scale sgd_alpha=1e-05 sgd_iter=1000 n_comp=600]2025-11-19 23:48:20,387 [INFO

In [1]:
# save results_df to csv
results_df.to_csv('svn_hyperparameter_optimisation.csv', index=False)

NameError: name 'results_df' is not defined

## Summary statistics

In [ ]:
metrics = ['best_val_loss', 'dev_accuracy', 'dev_auc']
fig, axes = plt.subplots(1, len(metrics), figsize=(6 * len(metrics), 4))
for ax, metric in zip(axes, metrics):
    sns.boxplot(data=results_df, y=metric, ax=ax)
    ax.set_title(metric)
plt.tight_layout()
plt.show()

results_df[['approximate', 'C', 'gamma', 'sgd_alpha', 'sgd_max_iter', 'n_components', 'best_val_loss', 'dev_accuracy', 'dev_auc']].head(10)


## Hyperparameter effects

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.scatterplot(data=results_df, x='C', y='dev_accuracy', hue='approximate', ax=axes[0])
sns.scatterplot(data=results_df, x='sgd_alpha', y='best_val_loss', hue='n_components', ax=axes[1])
sns.scatterplot(data=results_df, x='sgd_max_iter', y='dev_auc', hue='gamma', ax=axes[2])
for ax in axes:
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


## Inspect top runs

In [ ]:
top_runs = results_df.head(3)
for _, row in top_runs.iterrows():
    key = (
        f"approx={row['approximate']} C={row['C']} gamma={row['gamma']} "
        f"sgd_alpha={row['sgd_alpha']} sgd_iter={row['sgd_max_iter']} n_comp={row['n_components']}"
    )
    fig = plot_training_history(histories[key], title=key)
    plt.show()
